# 02 Time, Geography, And Vocabulary

Assignment steps 1-5: extract `D(t)`, `S(t)`, `Geo(t)`, title vocabulary, and global `V`.


In [1]:
import csv
import json
from pathlib import Path

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None

## Artifact Counts


In [2]:
{
    "D(t) table_times": parquet_count("data/processed/table_times.parquet"),
    "S(t) table_strings": parquet_count("data/processed/table_strings.parquet"),
    "Geo(t) table_geographies": parquet_count("data/processed/table_geographies.parquet"),
    "title_terms": parquet_count("data/processed/title_terms.parquet"),
    "term_occurrences": parquet_count("data/processed/term_occurrences.parquet"),
    "table_vocabulary": parquet_count("data/processed/table_vocabulary.parquet"),
    "global V vocabulary": parquet_count("data/processed/vocabulary.parquet"),
}

{'D(t) table_times': 26715,
 'S(t) table_strings': 93116,
 'Geo(t) table_geographies': 265594,
 'title_terms': 3356,
 'term_occurrences': 838333,
 'table_vocabulary': 34586,
 'global V vocabulary': 9571}

## Representative Evidence


In [3]:
examples = {}
for name, rel in {
    "time": "data/processed/table_times.parquet",
    "geography": "data/processed/table_geographies.parquet",
    "vocabulary": "data/processed/vocabulary.parquet",
}.items():
    path = ROOT / rel
    examples[name] = pq.read_table(path).slice(0, 3).to_pylist() if path.exists() else []
examples

{'time': [{'time_id': 'time_78751831e5f4edc81396',
   'table_id': 'aei_hri',
   'raw_value': '2011.0',
   'normalized_value': '2011',
   'start_date': '2011-01-01',
   'end_date': '2011-12-31',
   'granularity': 'year',
   'source_area': 'header_time',
   'location': 'header[0]',
   'source_span_start': None,
   'source_span_end': None,
   'extractor_rule': 'header_year',
   'confidence': 1.0,
   'run_id': 'run_efcf4769184deff19ad6'},
  {'time_id': 'time_a521187a0c4095ef55a7',
   'table_id': 'aei_hri',
   'raw_value': '2021.0',
   'normalized_value': '2021',
   'start_date': '2021-01-01',
   'end_date': '2021-12-31',
   'granularity': 'year',
   'source_area': 'header_time',
   'location': 'header[10]',
   'source_span_start': None,
   'source_span_end': None,
   'extractor_rule': 'header_year',
   'confidence': 1.0,
   'run_id': 'run_efcf4769184deff19ad6'},
  {'time_id': 'time_1df10185ecb4a5dd0831',
   'table_id': 'aei_hri',
   'raw_value': '2022.0',
   'normalized_value': '2022',
   

## Extraction Evaluation


In [4]:
metrics = read_json("report/extraction_metrics.json")
metrics.get("metrics", metrics)

{'failure_examples': [],
 'geography': {'f1': 1.0,
  'fn': 0,
  'fp': 0,
  'precision': 1.0,
  'recall': 1.0,
  'tn': 22,
  'tp': 97},
 'string': {'f1': 1.0,
  'fn': 0,
  'fp': 0,
  'precision': 1.0,
  'recall': 1.0,
  'tn': 0,
  'tp': 170},
 'time': {'f1': 1.0,
  'fn': 0,
  'fp': 0,
  'precision': 1.0,
  'recall': 1.0,
  'tn': 0,
  'tp': 128},
 'title': {'f1': 1.0,
  'fn': 0,
  'fp': 0,
  'precision': 1.0,
  'recall': 1.0,
  'tn': 0,
  'tp': 50}}